In [2]:
import torch
import numpy as np

class RotaryPositionEmbedding:
    """
    Implements Rotary Position Embeddings (RoPE) for self-attention, as described in the original paper.
    RoPE encodes absolute positional information while maintaining relative positional dependence.
    
    How RoPE Works:
    ----------------
    Unlike traditional positional encodings that add position information to embeddings, RoPE encodes positions
    by rotating the embedding vectors in a high-dimensional space. This is done using complex number arithmetic
    based on Euler's formula:
    
        e^(iθ) = cos(θ) + i*sin(θ)
    
    Instead of explicitly adding position embeddings, RoPE **rotates** the query and key vectors in self-attention,
    preserving their relative positional information while avoiding fixed absolute position dependence.
    
    The algorithm follows these steps:
    1. Precompute position-dependent rotation frequencies using a predefined base.
    2. Convert embeddings into complex numbers.
    3. Apply the rotation using complex multiplication.
    4. Convert the rotated embeddings back to real values.
    """
    def __init__(self, dim: int, num_heads: int, base: int = 10000):
        """
        Initializes RoPE with the given embedding dimension and number of attention heads.
        
        Args:
            dim (int): Embedding dimension (should be even and divisible by num_heads).
            num_heads (int): Number of attention heads.
            base (int): Base frequency for computing angles.
        """
        assert dim % num_heads == 0, "Embedding dimension must be divisible by number of heads"
        assert dim % 2 == 0, "Embedding dimension must be even"
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads  # Dimension per head
        self.base = base
        
    def precompute_theta_pos_frequencies(self, seq_len: int, device: str):
        """
        Precomputes the RoPE theta position-dependent frequencies.
        
        Args:
            seq_len (int): Sequence length.
            device (str): Device to store tensors.
        
        Returns:
            torch.Tensor: Complex exponential frequency tensor.
        """
        assert self.head_dim % 2 == 0, "Head dimension must be divisible by 2"
        
        # Build the theta parameter
        # According to the formula theta_i = 10000^(-2(i-1)/dim) for i = [1, 2, ... dim/2]
        theta_numerator = torch.arange(0, self.head_dim, 2).float()  # Shape: (head_dim / 2,)
        theta = 1.0 / (self.base ** (theta_numerator / self.head_dim)).to(device)  # Shape: (head_dim / 2,) or dim/2

        # Construct the positions (the "m" parameter)
        m = torch.arange(seq_len, device=device)  # Shape: (seq_len,)

        # Multiply each theta by each position using the outer product.
        # Shape: (Seq_Len) outer_product* (Head_Dim / 2) -> (Seq_Len, Head_Dim / 2)
        freqs = torch.outer( m, theta).float()  # Shape: (seq_len, head_dim / 2)

        # We can compute complex numbers in the polar form c = R * exp(m * theta), where R = 1 as follows:
        # (Seq_Len, Head_Dim / 2) -> (Seq_Len, Head_Dim / 2)
        freqs_complex = torch.polar(torch.ones_like(freqs), freqs)  # Shape: (seq_len, head_dim / 2)
        
        return freqs_complex
    
    def apply_rotary(self, x: torch.Tensor, freqs_complex: torch.Tensor) -> torch.Tensor:
        """
        Applies the RoPE transformation to the input tensor across multiple attention heads.
        
        Args:
            x (torch.Tensor): Input tensor of shape (batch, seq_len, num_heads, head_dim)
            freqs_complex (torch.Tensor): Precomputed RoPE rotation frequencies.
        
        Returns:
            torch.Tensor: Rotated tensor of the same shape.
        
        Steps:
        1. Convert input tensor into complex numbers.
        2. Perform complex multiplication to apply the rotation.
        3. Convert back to real values and restore original shape.
        """
        x_reshaped = x.float().reshape(*x.shape[:-1], -1, 2)  # (B, Seq_Len, H, Head_Dim/2, 2)
        x_complex = torch.view_as_complex(x_reshaped)  # Convert last two dims to a complex number
        freqs_complex = freqs_complex.unsqueeze(0).unsqueeze(2)  # (1, Seq_Len, 1, Head_Dim/2)
        x_rotated = x_complex * freqs_complex  # (B, Seq_Len, H, Head_Dim/2)
        x_out = torch.view_as_real(x_rotated)  # (B, Seq_Len, H, Head_Dim/2, 2)
        x_out = x_out.reshape(*x.shape)  # Restore original shape
        
        return x_out.type_as(x)

# Example usage
if __name__ == "__main__":
    batch_size, seq_len, embed_dim, num_heads = 2, 5, 8, 2  # Small example
    rope = RotaryPositionEmbedding(dim=embed_dim, num_heads=num_heads)
    
    # Random input tensor (batch, seq_len, num_heads, head_dim)
    x = torch.randn(batch_size, seq_len, num_heads, embed_dim // num_heads)
    freqs_complex = rope.precompute_theta_pos_frequencies(seq_len, device="cpu")
    x_rotated = rope.apply_rotary(x, freqs_complex)
    
    print("Original Tensor:", x)
    print("Rotated Tensor:", x_rotated)


Original Tensor: tensor([[[[ 1.9713,  0.8928,  1.1575, -1.3495],
          [ 0.2065,  0.8347,  1.2226,  0.6635]],

         [[-1.6249,  0.8622,  0.8298,  1.1808],
          [ 1.3835,  0.3106,  0.5108, -0.5280]],

         [[ 0.0312, -0.3627, -0.1800,  0.5639],
          [-1.2126, -1.0511,  0.3395,  0.1623]],

         [[ 1.5915,  0.3992, -0.2579,  1.9232],
          [-0.3461, -0.1262, -0.3133, -2.5844]],

         [[-1.3849, -1.4356,  0.0893, -0.1340],
          [-0.3042, -0.7274,  0.4263,  0.1754]]],


        [[[-0.5232,  0.2868,  0.6013,  0.6596],
          [-0.2407,  1.4851,  0.0556, -0.8819]],

         [[ 0.8920, -0.3136,  0.0712, -0.2120],
          [ 1.4956,  1.4517, -1.1896,  0.8562]],

         [[ 1.2301, -0.5400, -0.5548,  0.5649],
          [-1.0477, -0.4024,  0.6332,  0.1259]],

         [[-1.3787,  0.6860,  2.0891,  0.7126],
          [-1.1845,  3.0495,  0.9056, -0.7709]],

         [[ 1.1967, -0.3252,  0.3496, -0.8866],
          [ 0.1979, -0.7718, -1.4344,  0.3661]]]])
